# nb_10_gold_premium_revenue

**Layer**: Gold | **Source**: `lh_silver_insurance` | **Target**: `lh_gold_insurance.gold_premium_revenue`

**Purpose**: Aggregate premium revenue by policy type, billing period, and payment status.

**Output Columns**:
| Column | Description |
|---|---|
| policy_type | Type of insurance policy |
| billing_period | Billing quarter (e.g., 2024-Q3) |
| payment_status | Status of premium payment |
| premium_count | Number of premium records |
| total_amount_due | Sum of amounts due |
| total_amount_paid | Sum of amounts paid |
| collection_rate_pct | Percentage of amount due that was collected |
| avg_premium_amount | Average premium amount |

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, round as _round,
    current_timestamp, when, lit
)

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_10_gold_premium_revenue").getOrCreate()

print("nb_10_gold_premium_revenue - Gold Layer Aggregation")

In [ ]:
# ============================================================
# Step 1: Read Silver tables
# ============================================================
df_premiums = spark.table("premiums")
df_policies = spark.table("policies")

print(f"Silver premiums: {df_premiums.count():,} rows")
print(f"Silver policies: {df_policies.count():,} rows")

In [ ]:
# ============================================================
# Step 2: Join premiums with policy type
# ============================================================
df_enriched = df_premiums.join(
    df_policies.select("policy_id", "policy_type"),
    "policy_id",
    "left"
).withColumn("policy_type", when(col("policy_type").isNull(), lit("unknown")).otherwise(col("policy_type")))

print(f"Enriched premiums: {df_enriched.count():,} rows")

In [ ]:
# ============================================================
# Step 3: Aggregate by policy_type, billing_period, payment_status
# ============================================================
df_gold = df_enriched.groupBy("policy_type", "billing_period", "payment_status").agg(
    count("premium_id").alias("premium_count"),
    _round(_sum("amount_due"), 2).alias("total_amount_due"),
    _round(_sum("amount_paid"), 2).alias("total_amount_paid"),
    _round(avg("amount_due"), 2).alias("avg_premium_amount")
).withColumn(
    "collection_rate_pct",
    _round(
        when(col("total_amount_due") > 0,
             col("total_amount_paid") / col("total_amount_due") * 100
        ).otherwise(0.0), 1
    )
).orderBy("policy_type", "billing_period", "payment_status")

gold_count = df_gold.count()
print(f"Gold premium revenue: {gold_count:,} aggregated rows")
df_gold.show(10, truncate=False)

In [ ]:
# ============================================================
# Step 4: Write to Gold
# ============================================================
df_gold.withColumn("_processed_at", current_timestamp()) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_premium_revenue")

print(f"✓ {gold_count:,} rows written to gold_premium_revenue")

In [ ]:
# ============================================================
# Validation
# ============================================================
readback = spark.table("gold_premium_revenue").count()
assert readback == gold_count
print(f"✓ Verification passed: {readback:,} rows")

# Revenue summary by policy type
print("\n--- Revenue by Policy Type ---")
spark.sql("""
    SELECT policy_type,
           SUM(premium_count) as total_premiums,
           ROUND(SUM(total_amount_due), 2) as total_due,
           ROUND(SUM(total_amount_paid), 2) as total_paid,
           ROUND(SUM(total_amount_paid) / NULLIF(SUM(total_amount_due), 0) * 100, 1) as collection_rate
    FROM gold_premium_revenue
    GROUP BY policy_type
    ORDER BY total_due DESC
""").show(truncate=False)